# Step 0B — Source Investigation

## Objective

Investigate the candidate maritime data sources before designing the ingestion
pipeline or canonical schema.

For each source:

- identify the source owner and access location
- determine file format and publication cadence
- obtain a representative sample
- inspect the original schema
- determine the grain of the data
- identify vessel identifiers
- identify important business fields
- profile nulls, duplicates, and data types
- document parsing or schema concerns
- note fields that may map to the final canonical vessel model

## Source Investigation Order

1. USCG Merchant Vessels of the United States
2. USCG PSIX / CGMIX
3. NOAA / MarineCadastre AIS

The findings from this notebook will be used in Step 0C to create the
source-to-target mapping and initial Gold schema.

### 1. USCG Merchant Vessels of the United States

**Organization:** U.S. Coast Guard

**Purpose:** Public vessel registry containing vessel identity,
characteristics, ownership, service, and related information.

**Candidate formats:**
- Fixed-width text
- Tab-delimited text
- Excel
- Microsoft Access

**Initial questions:**
- What does one record represent?
- How frequently is the dataset published?
- Is each release a full snapshot or incremental update?
- Which vessel identifiers are consistently populated?
- What is the original source schema?
- Which fields overlap with PSIX and AIS?
- Are field widths and data types stable across releases?
- How should weekly processing handle a source published less frequently?

In [1]:
import requests

url = "https://www.dco.uscg.mil/Our-Organization/Assistant-Commandant-for-Prevention-Policy-CG-5P/Inspections-Compliance-CG-5PC-/Office-of-Investigations-Casualty-Analysis/Merchant-Vessels-of-the-United-States/"

response = requests.get(url, timeout=30)

print(response.status_code)
print(response.url)

403
https://www.dco.uscg.mil/Our-Organization/Assistant-Commandant-for-Prevention-Policy-CG-5P/Inspections-Compliance-CG-5PC-/Office-of-Investigations-Casualty-Analysis/Merchant-Vessels-of-the-United-States/


Observed record width: 2,568 characters per row in the September 2026 fixed-width file.

In [5]:
# manually loaded txt files
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")
VESDOC_FILE = RAW_DATA_DIR / "vesdocSep26Rfixed.txt"

with VESDOC_FILE.open("r", encoding="utf-8", errors="replace") as f:
    for i in range(3):
        line = f.readline().rstrip("\r\n")
        print(i + 1, len(line), repr(line[:120]))

1 2568 '328660    $5 COVER                                                  947092                                  CCHEX211A989'
2 2568 '1581305   $AVE TOMORROW                                             1297456                                             '
3 2568 '383024    $CENT OF A WOMAN                                          989695                                  TNRD1875C191'


Data Fields in README

In [7]:
colspecs = [
    (0, 10),    # VesselID
    (10, 60),   # VesselName
    (60, 68),   # Call_Sign
    (68, 78),   # Official_Number
    (78, 108),  # IMO_Number
    (108, 138), # Hull_Number
    (138, 168), # HIN
    (168, 198), # Service
    (198, 200), # Flag
    (200, 201), # Self_Propelled_Ind
    (201, 208), # Gross_Ton
    (208, 215), # Net_Ton
    (215, 222), # Length
    (222, 228), # Breadth
    (228, 234), # Depth
]

In [9]:
import pandas as pd

names = [
    "VesselID",
    "VesselName",
    "Call_Sign",
    "Official_Number",
    "IMO_Number",
    "Hull_Number",
    "HIN",
    "Service",
    "Flag",
    "Self_Propelled_Ind",
    "Gross_Ton",
    "Net_Ton",
    "Length",
    "Breadth",
    "Depth",
]

df_sample = pd.read_fwf(
    VESDOC_FILE,
    colspecs=colspecs,
    names=names,
    nrows=20
)

df_sample

,VesselID,VesselName,Call_Sign,Official_Number,IMO_Number,Hull_Number,HIN,Service,Flag,Self_Propelled_Ind,Gross_Ton,Net_Ton,Length,Breadth,Depth
0,328660,$5 COVER,NaN,947092,NaN,CCHEX211A989,NaN,Recreational,US,1,14.0,11,31.7,11.8,5.9
1,1581305,$AVE TOMORROW,NaN,1297456,NaN,NaN,SP332079E920,Recreational,US,1,10.0,10,32.0,9.0,5.3
2,383024,$CENT OF A WOMAN,NaN,989695,NaN,TNRD1875C191,NaN,Recreational,US,1,16.0,12,41.8,8.3,7.0
3,256860,$MEGA-BUCK$,NaN,681310,NaN,NaN,NaN,Recreational,US,1,10.0,8,28.7,10.5,5.2
4,1359771,& BEYOND,NaN,1262935,NaN,VKY54532L910,NaN,Recreational,US,1,63.0,51,54.0,17.3,10.2
5,593322,#4 PLAY,NaN,1133158,NaN,NaN,USDC90SUL102,Recreational,US,1,9.0,7,28.5,9.8,5.0
6,461269,#44395,NaN,1230524,NaN,44395,NaN,Recreational,US,1,12.0,9,44.2,12.7,3.3
7,576732,#7,NaN,1129932,NaN,E-01-7,NaN,Industrial Vessel,US,0,504.0,504,150.0,50.0,8.0
8,938881,#WINNING II,NaN,1209179,NaN,SERF1276G607,NaN,Recreational,US,1,26.0,21,45.0,14.0,6.3
9,645561,$$ GONE WAY TOO FAR $$,NaN,1146665,NaN,NaN,RGMVA501D304,Recreational,US,1,12.0,10,32.0,11.2,5.4


### USCG documented data types

| Field | USCG type | Length | Meaning |
|---|---|---:|---|
| VesselID | AN | 10 | Alphanumeric |
| VesselName | AN | 50 | Alphanumeric |
| Call_Sign | AN | 8 | Alphanumeric |
| Official_Number | AN | 10 | Alphanumeric |
| IMO_Number | AN | 30 | Alphanumeric |
| Hull_Number | AN | 30 | Alphanumeric |
| HIN | AN | 30 | Alphanumeric |
| Service | CH | 30 | Character |
| Flag | CH | 2 | Character |
| Self_Propelled_Ind | Logical | 1 | 1 / 0 / blank |
| Gross_Ton | DS | 7 | Decimal |
| Net_Ton | DS | 7 | Decimal |
| Length | DS | 7 | Decimal |
| Breadth | DS | 6 | Decimal |
| Depth | DS | 6 | Decimal |

**USCG type definitions**

- `AN` — Alphanumeric
- `CH` — Character
- `DS` — Decimal
- `IS` — Integer
- `Logical` — `1` / `0`
- `CM` — `"X"` / blank

In [11]:
dtype_table = (
    df_sample.dtypes
    .astype(str)
    .rename("pandas_dtype")
    .reset_index()
    .rename(columns={"index": "field"})
)

dtype_table

,field,pandas_dtype
0,VesselID,int64
1,VesselName,str
2,Call_Sign,str
3,Official_Number,int64
4,IMO_Number,float64
5,Hull_Number,str
6,HIN,str
7,Service,str
8,Flag,str
9,Self_Propelled_Ind,int64


In [12]:
# compare readme to actual txt file types
uscg_types = {
    "VesselID": "AN",
    "VesselName": "AN",
    "Call_Sign": "AN",
    "Official_Number": "AN",
    "IMO_Number": "AN",
    "Hull_Number": "AN",
    "HIN": "AN",
    "Service": "CH",
    "Flag": "CH",
    "Self_Propelled_Ind": "Logical",
    "Gross_Ton": "DS",
    "Net_Ton": "DS",
    "Length": "DS",
    "Breadth": "DS",
    "Depth": "DS",
}

comparison = dtype_table.copy()

comparison["uscg_documented_type"] = comparison["field"].map(uscg_types)

comparison = comparison[
    ["field", "uscg_documented_type", "pandas_dtype"]
]

comparison

,field,uscg_documented_type,pandas_dtype
0,VesselID,AN,int64
1,VesselName,AN,str
2,Call_Sign,AN,str
3,Official_Number,AN,int64
4,IMO_Number,AN,float64
5,Hull_Number,AN,str
6,HIN,AN,str
7,Service,CH,str
8,Flag,CH,str
9,Self_Propelled_Ind,Logical,int64


In [13]:
canonical_types = {
    "VesselID": "string",
    "VesselName": "string",
    "Call_Sign": "string",
    "Official_Number": "string",
    "IMO_Number": "string",
    "Hull_Number": "string",
    "HIN": "string",
    "Service": "string",
    "Flag": "string",
    "Self_Propelled_Ind": "boolean",
    "Gross_Ton": "decimal/double",
    "Net_Ton": "decimal/double",
    "Length": "decimal/double",
    "Breadth": "decimal/double",
    "Depth": "decimal/double",
}

comparison["proposed_canonical_type"] = (
    comparison["field"].map(canonical_types)
)

comparison

,field,uscg_documented_type,pandas_dtype,proposed_canonical_type
0,VesselID,AN,int64,string
1,VesselName,AN,str,string
2,Call_Sign,AN,str,string
3,Official_Number,AN,int64,string
4,IMO_Number,AN,float64,string
5,Hull_Number,AN,str,string
6,HIN,AN,str,string
7,Service,CH,str,string
8,Flag,CH,str,string
9,Self_Propelled_Ind,Logical,int64,boolean
